In [0]:
import requests

APP_ID = "c3a0d2c4"
APP_KEY = "68f04bb76ce001937f2ba0d6a88d1aa3"

url = "https://api.adzuna.com/v1/api/jobs/us/search/1"

params = {
    "app_id": APP_ID,
    "app_key": APP_KEY,
    "results_per_page": 50,
    "what": "data engineer",
    "where": "Texas"
}

response = requests.get(url, params=params)

print("Status:", response.status_code)

response.raise_for_status()

data = response.json()

jobs = data["results"]

print("Jobs retrieved:", len(jobs))

In [0]:
job = jobs[0]
print(job.keys())

In [0]:
print(job["company"])
print(job["location"])
print(job["category"])

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    BooleanType
)

from pyspark.sql.functions import current_timestamp


schema = StructType([
    StructField("job_id", StringType(), True),
    StructField("title", StringType(), True),
    StructField("description", StringType(), True),
    StructField("company", StringType(), True),
    StructField("location", StringType(), True),
    StructField("category", StringType(), True),
    StructField("salary_min", DoubleType(), True),
    StructField("salary_max", DoubleType(), True),
    StructField("salary_is_predicted", BooleanType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("job_url", StringType(), True),
    StructField("created", StringType(), True)
])


rows = []

for job in jobs:

    rows.append((
        str(job.get("id")) if job.get("id") else None,

        job.get("title"),

        job.get("description"),

        job.get("company", {}).get("display_name"),

        job.get("location", {}).get("display_name"),

        job.get("category", {}).get("display_name"),

        float(job["salary_min"])
        if job.get("salary_min") is not None
        else None,

        float(job["salary_max"])
        if job.get("salary_max") is not None
        else None,

        bool(job["salary_is_predicted"])
        if job.get("salary_is_predicted") is not None
        else None,

        float(job["latitude"])
        if job.get("latitude") is not None
        else None,

        float(job["longitude"])
        if job.get("longitude") is not None
        else None,

        job.get("redirect_url"),

        job.get("created")
    ))


jobs_df = spark.createDataFrame(
    rows,
    schema=schema
)


jobs_df = jobs_df.withColumn(
    "pipeline_run_time",
    current_timestamp()
)


display(jobs_df)

In [0]:
jobs_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "career_os.bronze.jobs_raw"
    )